# 16 - D22: maize field photos and the class balance

**In short:** D17 added 3,877 field photos, but only 108 were maize and none northern leaf
blight, so training became potato-heavy and maize did not improve. This notebook checks
maize field photos from South Africa and adds them to the **training data only**, as a new
list `train_d22.csv` (`train_d17.csv` unchanged plus the new photos). It then shows how many
photos every class, crop and source has, so **whether to weight the loss is decided from
the counts, before any training** (notebook 17).

| Source | Classes used | Licence |
|---|---|---|
| Craze (2022), *Diseases of maize in the field*, University of Pretoria (Figshare 20237613): 2,355 field photos from different times and places in South Africa | gray leaf spot, northern leaf blight, common rust, healthy | CC BY 4.0 |

**Which photos are used (fixed before seeing any photo, `src/data/d22.py`):** a photo can
carry several labels, so a photo is used only if it shows **exactly one of our maize
classes and no disease that is not ours** (phaeosphaeria leaf spot, southern rust, an
unidentified disease). Insect damage or tearing ("Other") is allowed on a diseased leaf, as
real field leaves look, but not on a healthy one, where it could look like a symptom.

**Checks:** the licence is read from Figshare; both files are verified against Figshare's
MD5 checksums. A photo is dropped if it matches any existing photo (the original training
sources, PlantDoc, the selection set **and D17's added photos**), exactly or as a
near-duplicate, also when turned or mirrored; near-copies inside the new photos are reduced
to one. **You then look at a sheet of sample photos per class before anything is saved.**

No GPU needed. About 30-45 minutes: the archive is 10.8 GB and goes to Colab's own disk,
not Drive; only the usable photos are unpacked, and only the kept ones are saved to Drive,
shrunk (well under 100 MB).

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Licence and download

Both files go to Colab's disk only and are checked against Figshare's MD5 checksums.

In [ ]:
import importlib
import shutil

import pandas as pd

from src.data import cleaning, d17, d22, selection_set
for module in (cleaning, selection_set, d17, d22):
    importlib.reload(module)

PHOTOS_DIR = config.RAW_DATA_DIR / "d22_photos"   # kept photos, on Drive (written at the end)
LOCAL = Path("/content/d22")                      # download and unpacked photos, Colab disk only

# Nothing earlier is touched: this notebook writes only these places.
WRITES = [PHOTOS_DIR, config.SPLITS_DIR / "train_d22.csv", config.METADATA_DIR / "d22_sources.csv",
          config.METADATA_DIR / "d22_manifest.csv"]
for earlier in ("train.csv", "train_d17.csv"):
    assert config.SPLITS_DIR / earlier not in WRITES
assert not any(config.MODELS_DIR in p.parents for p in WRITES)

rec = d22.record()
licence = d22.check_licence(rec)
records = d22.download(rec, LOCAL / "download")
sources_table = pd.DataFrame([{**r, "licence": licence, "dataset": f"Figshare {d22.ARTICLE}",
                               "citation": d22.CITATION} for r in records])
display(sources_table[["file", "licence", "bytes", "md5", "how"]])

## Which photos are used

**Check this table before going on.** On 22 Oct the label file gave, before the duplicate
checks: gray leaf spot 713, healthy 232, northern leaf blight 170, common rust 119. Every
other photo is listed with the reason it is left out. Only the usable photos are unpacked.

In [ ]:
labels = pd.read_csv(LOCAL / "download" / d22.LABELS_FILE)
labelled = d22.label_rows(labels)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)
display(d22.label_summary(labelled))

extracted = d22.extract_mapped(LOCAL / "download" / d22.IMAGES_FILE, labelled, LOCAL / "unpacked")
print(f"{extracted.attrs['archive_photos']} photos in the archive; "
      f"{extracted.attrs['archive_photos_not_in_labels']} of them are not in {d22.LABELS_FILE} (not used).")
problems = extracted[extracted["label_reason"].str.contains("archive")]
print(f"Mapped photos that could not be matched in the archive: {len(problems)}")
display(problems[["file", "label_reason"]].head(20))

inventory = d22.inventory(extracted)
print("Unpacked and mapped:", inventory["class"].value_counts().to_dict())

## Duplicate checks

Existing photos: every original training-source and PlantDoc photo (`clean_manifest.csv`,
notebook 06), every selection-set photo (`selection_manifest.csv`, notebook 12) and every
photo D17 added (`d17_manifest.csv`, notebook 14).

In [ ]:
d17_photos = pd.read_csv(config.METADATA_DIR / "d17_manifest.csv", usecols=["md5", "phash", "keep"])
existing = pd.concat([
    pd.read_csv(config.METADATA_DIR / "clean_manifest.csv", usecols=["md5", "phash"]),
    pd.read_csv(config.METADATA_DIR / "selection_manifest.csv", usecols=["md5", "phash"]),
    d17_photos[d17_photos["keep"].astype(str) == "True"][["md5", "phash"]],
], ignore_index=True)
print(f"Compared against {len(existing)} existing photos")

mapped = inventory[inventory["class"].notna()].reset_index(drop=True)
prints = selection_set.fingerprint_all(list(mapped["path"]))
checked = pd.concat([mapped, prints], axis=1)
unmapped = inventory[inventory["class"].isna()].assign(error=None, md5=None, phash=None, phash_variants=None,
                                                       width=None, height=None)
result = selection_set.deduplicate(pd.concat([checked, unmapped], ignore_index=True), existing,
                                   config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)
# A copy of a matched photo that was changed a little more must not slip through on its own.
result = d17.drop_copies_of_matches(result, config.NEAR_DUPLICATE_MAX_DISTANCE)
result["reason"] = result["reason"].replace("near_duplicate_within_selection", "near_duplicate_within_new_photos")

reasons = (result.assign(reason=result["reason"].replace("", "kept"))
           .groupby("reason").size().rename("photos").reset_index())
display(reasons)
kept = result[result["keep"]]
display(kept["class"].value_counts().rename("kept").to_frame())

## STOP: look at the photos

One sheet per class, 18 random kept photos each. Check that each sheet shows **real field
photos of maize leaves** with that condition:

- **gray leaf spot:** long, narrow, rectangular grey or tan spots that stop at the leaf veins;
- **northern leaf blight:** long, cigar-shaped grey-green or tan lesions, several cm long;
- **common rust:** many small, raised, brick-red to brown pustules, often on both sides;
- **healthy:** a green leaf with no spots (small tears are fine).

If a sheet looks wrong (another crop, drawings, mostly a different disease), stop and
describe it to Claude. Nothing has been saved yet.

In [ ]:
REVIEW_DIR = LOCAL / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sheets = []
for (name, cls), paths in d17.review_sample(kept, 18, config.RANDOM_SEED).items():
    total = int((kept["class"] == cls).sum())
    sheet = d17.contact_sheet(paths, f"{name} | {cls} | {total} kept")
    target = REVIEW_DIR / f"{name}__{cls.replace(' ', '_').replace('(', '').replace(')', '')}.jpg"
    sheet.save(target, quality=85)
    sheets.append(target)
    display(sheet)

In [ ]:
# After looking at EVERY sheet above, change False to True and run this cell.
I_CHECKED_THE_PHOTOS = False

assert I_CHECKED_THE_PHOTOS, "Look at every sheet above first, then set I_CHECKED_THE_PHOTOS = True."
print("Checked. Saving next.")

## Save the kept photos and `train_d22.csv`

Photos are saved to Drive, shrunk to a shorter side of 256 pixels (as every other split),
as `raw/d22_photos/<class>/<file>`. `train_d22.csv` is `train_d17.csv` unchanged plus one row
per new photo, with the same columns.

In [ ]:
# A rerun keeps a different set of photos, so this notebook's own earlier copies go first.
if PHOTOS_DIR.exists():
    shutil.rmtree(PHOTOS_DIR)

rows, jobs = [], []
for i, row in kept.reset_index(drop=True).iterrows():
    safe = f"{i:05d}_{Path(row['path']).name}".replace(" ", "_")
    target = PHOTOS_DIR / row["class"] / safe
    jobs.append((Path(row["path"]), target))
    rows.append({"path": str(target.relative_to(config.RAW_DATA_DIR)), "class": row["class"], "source": row["source"],
                 "crop": row["crop"], "dup_group": int(row["dup_group"]), "md5": row["md5"]})
selection_set.save_all_small(jobs)
added = pd.DataFrame(rows)

train = pd.read_csv(config.SPLITS_DIR / "train.csv")
train_d17 = pd.read_csv(config.SPLITS_DIR / "train_d17.csv")
train_d22 = d22.build_train_d22(train_d17, added)
train_d22.to_csv(config.SPLITS_DIR / "train_d22.csv", index=False)
sources_table.to_csv(config.METADATA_DIR / "d22_sources.csv", index=False)
result.drop(columns=["path"]).assign(path=result["path"].astype(str)).to_csv(
    config.METADATA_DIR / "d22_manifest.csv", index=False)
print(f"{len(added)} photos added; train_d17.csv {len(train_d17)} rows (unchanged) -> train_d22.csv {len(train_d22)} rows")

## The class balance: read this before deciding on weights

Training photos per class in each list (`train.csv`, `train_d17.csv`, `train_d22.csv`):

- **share of last (%)**: each class's share of `train_d22.csv`;
- **times the smallest**: how many times larger than the smallest class;
- **balanced weight (preview)**: the weight each class *would* get if the loss were
  weighted (total / (classes x class photos)), and the same capped at 5. **Preview only:
  nothing is weighted unless we decide so from these counts, before notebook 17.**

Then photos per crop, and where each class's photos come from (source).

In [ ]:
lists = {"train.csv": train, "train_d17.csv": train_d17, "train_d22.csv": train_d22}
balance = d22.class_balance(lists)
display(balance)
crops = pd.concat({name: d22.crop_balance(df) for name, df in lists.items()}, axis=1)
display(crops)
sources = d22.source_balance(train_d22)
display(sources)

## Copy the summaries into the repository

In [ ]:
REPO_RESULTS = ML_DIR / "results" / "d22_maize_data"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
sources_table.to_csv(REPO_RESULTS / "sources.csv", index=False)
d22.label_summary(labelled).to_csv(REPO_RESULTS / "labels.csv", index=False)
reasons.to_csv(REPO_RESULTS / "checks.csv", index=False)
balance.to_csv(REPO_RESULTS / "class_balance.csv")
crops.to_csv(REPO_RESULTS / "crop_balance.csv")
sources.to_csv(REPO_RESULTS / "source_balance.csv")
(REPO_RESULTS / "review_sheets").mkdir(exist_ok=True)
for sheet in sheets:
    shutil.copy(sheet, REPO_RESULTS / "review_sheets" / sheet.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

In [ ]:
# Writes everything to Drive. If this takes very long, check Drive is not full.
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S); only the results folder is committed, to a new branch.

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "16_d22_maize_field_data.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/d22-maize-data",
    message="D22: check South African maize field photos and write train_d22.csv",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))